# 🧠 Template 2 : modèle simple (embeddings + mean pooling) — AT&T Spam Detector

*Préparé par Claude le 19/10/2026. Aucun code de solution ici : c'est toi qui écris.*

L'énoncé conseille **« Start simple »**. Le modèle le plus simple qui lit du texte, tu l'as déjà vu : c'est celui de la leçon
**M05-D05 `01-Train_text_classifier_with_embeddings`** (les avis de ShopSense). Ce template, c'est **la même recette** sur les SMS d'AT&T.

| Leçon ShopSense (M05-D05) | Ce template |
|---|---|
| Step 1. Get the data (20 avis écrits à la main) | Étape 1 : les 3 CSV du template 1 |
| Step 2. Vocabulary + encode + pad | Étapes 2 et 3 |
| *(tout en un seul paquet)* | Étape 4 : des lots de 64 avec `DataLoader` (M05-D03) |
| Step 3. Build the model (`SentimentNet`) | Étape 5 : `SpamNet`, la même classe |
| Step 4. Train | Étapes 6 et 7, avec la validation (M05-D03) |
| Step 5. Check the accuracy | Étape 8 : précision, rappel, F1 (l'accuracy ne suffit pas, cf. template 1) |
| — | Étape 9 : **l'expérience `<unk>`**, un piège que la leçon ne montre pas |
| Step 6. Predict / Bonus | Étapes 10 à 12 : erreurs, mots qui pèsent, sauvegarde |

**Avant de commencer** : il faut `src/train.csv`, `src/val.csv` et `src/test.csv`, écrits par l'étape 5 du template 1.

## Comment s'en servir

Même rythme que le template 1 : **🎯 Objectif**, **🧰 Outils** (avec la colonne **Vu où** : 📍 leçon, 🐍 Python, 🆕 hors cours),
**✍️ À toi**, **✅ Contrôle**. Les noms de variables imposés sont ceux qu'attendent les contrôles.

⚠️ **Le test ne sert pas ici.** On le prépare (étape 3) et on enregistre les probabilités du modèle dessus (étape 12),
mais on ne **regarde** ses scores qu'une fois, dans le template 4, pour comparer les modèles. Tous les réglages se font sur le **val**.

---
# Étape 0 — Imports et réglages
Rien à écrire : lance les deux cellules. Tous les réglages du modèle sont dans la 2ᵉ : c'est le seul endroit à modifier pour une expérience (étape 9).

In [ ]:
import copy

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from pathlib import Path
from sklearn.metrics import ConfusionMatrixDisplay, f1_score, precision_score, recall_score

DOSSIER_SRC = Path("src")
DOSSIER_MODELES = Path("models")
DOSSIER_MODELES.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", None)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("calcul sur :", device)

In [ ]:
# ── Réglages (hyperparamètres) ─────────────────────────────────────────
SEED = 42
MAX_LEN = 40          # nombre de mots gardés par SMS (ton choix du template 1, étape 3.c)
EMBEDDING_DIM = 32    # taille du vecteur de chaque mot (16 dans la leçon)
BATCH_SIZE = 64       # nombre de SMS par lot
LEARNING_RATE = 0.005
EPOCHS = 30
UNK_ID = 1            # numéro donné aux mots inconnus : 1 = <unk>, comme la leçon. Expérience de l'étape 9 : 0.

torch.manual_seed(SEED)
print("réglages prêts")

---
# Étape 1 — Charger les trois paquets

🎯 Relire les CSV du template 1 et sortir, pour chaque paquet, la **liste des textes** et la **liste des étiquettes**, comme `train_texts` et `train_labels` dans la leçon.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `pd.read_csv(chemin)` | lit un CSV | — | 📍 M02-D01 |
| `df["col"].tolist()` | colonne pandas → liste Python | — | 📍 M04-D02 Linear regression template, M05-D04 |

In [ ]:
# ✍️ À toi
# 1. train_df, val_df, test_df = les 3 CSV de DOSSIER_SRC
# 2. train_texts, train_labels = les colonnes text et label de train_df, en listes
#    pareil pour val_texts, val_labels et test_texts, test_labels
# 3. affiche la taille de chaque paquet et un exemple (texte -> étiquette), comme la leçon

In [ ]:
# ✅ Contrôle
assert (len(train_texts), len(val_texts), len(test_texts)) == (3617, 776, 776)
assert sum(train_labels) == 457 and sum(val_labels) == 98 and sum(test_labels) == 98, "nombre de spams inattendu"
assert isinstance(train_texts, list) and isinstance(train_labels, list)
print("✅ étape 1")

---
# Étape 2 — Le vocabulaire

🎯 Donner un numéro à chaque mot, **exactement comme la cellule 8 de la leçon** : `0` pour `<pad>`, `1` pour `<unk>`, puis les vrais mots à partir de 2.

Deux différences avec la leçon :
1. **Les mots sont mis en minuscules** avant le découpage : sinon `FREE`, `Free` et `free` auraient 3 numéros. Les avis de la leçon étaient déjà en minuscules.
2. **Le vocabulaire est construit sur le train seulement.** C'est la règle du `fit` sur le train seulement (le `StandardScaler` de M04) : le modèle ne doit rien savoir du val ni du test. Les mots qu'il n'a jamais vus deviendront `<unk>`, comme pour un vrai SMS qui arrive demain.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `texte.lower()` | met un texte en minuscules | — | 🐍 Python |
| `texte.split()` | découpe aux espaces → liste de mots | — | 🐍 Python, 📍 M05-D05 cellule 8 |
| `sorted(set(w for text in ... for w in text.split()))` | tous les mots différents, rangés | les deux `for` à la suite = « pour chaque texte, pour chaque mot » | 📍 M05-D05 cellule 8, M05-D03 Pythonic Python |
| `word_to_id[word] = len(word_to_id)` | ajoute un mot avec le numéro suivant | — | 📍 M05-D05 cellule 8 |

In [ ]:
# ✍️ À toi
# 1. all_words = tous les mots différents de train_texts, en minuscules, rangés
# 2. word_to_id = {"<pad>": 0, "<unk>": 1}, puis chaque mot de all_words avec le numéro suivant
# 3. vocab_size = la taille de word_to_id ; affiche-la

In [ ]:
# ✅ Contrôle
assert word_to_id["<pad>"] == 0 and word_to_id["<unk>"] == 1
assert "free" in word_to_id and "FREE" not in word_to_id, "les mots doivent être en minuscules"
assert vocab_size == len(word_to_id) == 10759, f"10759 attendus, obtenu {len(word_to_id)}"
print("✅ étape 2")

✍️ **À écrire** : la leçon avait un vocabulaire d'une cinquantaine de mots ; ici, combien ? Pourquoi autant, pour seulement 3 617 SMS ? (Pense à l'écriture SMS : `u`, `ur`, `2moro`…)

*Ta réponse :*

---
# Étape 3 — Encoder et mettre à la même longueur

🎯 Transformer chaque SMS en une ligne de `MAX_LEN` numéros, puis empiler les lignes dans un tenseur.

La leçon complète chaque avis avec des `0` jusqu'à la longueur du plus long (`pad`). Ici, le plus long SMS fait 171 mots : on **coupe** à `MAX_LEN` (ton choix du template 1), et on **complète** les plus courts.

```
"free entry in 2 a wkly comp"   →   [4051, 3595, 5004, 526, 1104, 10326, 2664, 0, 0, ... 0]
                                     └────── 7 numéros de mots ──────┘└── des 0 jusqu'à MAX_LEN ──┘
```

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `word_to_id.get(w, UNK_ID)` | le numéro du mot, ou `UNK_ID` s'il est inconnu | le 2ᵉ argument = la valeur si la clé manque | 📍 M05-D05 cellule 8 (`.get(w, 1)`) |
| `liste[:MAX_LEN]` | garde les `MAX_LEN` premiers éléments | — | 🐍 Python (tranches) |
| `[0] * n` | une liste de `n` zéros | — | 📍 M05-D05 cellule 8 |
| `torch.tensor(listes)` | liste de listes → tenseur 2D | `dtype=torch.float32` pour les étiquettes | 📍 M05-D05 cellule 8 |

⚠️ L'encodage doit mettre le texte **en minuscules**, comme le vocabulaire.

In [ ]:
# ✍️ À toi
# 1. def encode(text): liste des numéros des mots de text (en minuscules), UNK_ID pour un mot inconnu
# 2. def pad(encoded_list): pour chaque liste de numéros, couper à MAX_LEN puis compléter avec des 0
#    jusqu'à MAX_LEN ; renvoyer le tout en tenseur
# 3. X_train, X_val, X_test = pad de l'encodage de chaque texte
#    y_train, y_val, y_test = les étiquettes en tenseurs float32
# 4. affiche la forme de X_train et le premier SMS encodé

In [ ]:
# ✅ Contrôle
assert X_train.shape == (3617, MAX_LEN) and X_val.shape == (776, MAX_LEN) and X_test.shape == (776, MAX_LEN)
assert X_train.dtype == torch.int64 and y_train.dtype == torch.float32
assert X_train.max() < vocab_size and X_train.min() >= 0
assert (X_train == 1).sum() == 0, "aucun mot du train ne peut être inconnu : le vocabulaire vient du train"
part_inconnus = ((X_val == UNK_ID) & (X_val != 0)).sum().item() / max(1, (X_val != 0).sum().item()) if UNK_ID else None
if UNK_ID == 1:
    assert 0.11 < part_inconnus < 0.14, f"part de mots inconnus dans le val : {part_inconnus:.3f}"
    print(f"mots inconnus dans le val : {part_inconnus:.1%}")
print("✅ étape 3")

👀 **Observe** : dans la leçon, tous les mots du test étaient connus. Ici, environ **1 mot sur 8** du val est inconnu. Garde ce chiffre en tête : il revient à l'étape 9.

---
# Étape 4 — Les lots (`DataLoader`)

🎯 La leçon envoie ses 20 avis d'un coup. Avec 3 617 SMS, on les envoie **par lots de 64**, mélangés à chaque époque, comme dans **M05-D03 `02-Neural_networks_with_pytorch`**.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `data.TensorDataset(X, y)` | colle entrées et étiquettes ligne à ligne | — | 📍 M05-D03 Neural networks with PyTorch |
| `data.DataLoader(dataset, ...)` | distribue le dataset par lots | `batch_size=BATCH_SIZE`, `shuffle=True` **pour le train seulement** | 📍 M05-D03 |

Pourquoi pas `shuffle=True` pour le val ? On ne s'entraîne pas dessus : l'ordre ne change rien, et un ordre fixe permet de retrouver chaque SMS à l'étape 10.

In [ ]:
# ✍️ À toi
# 1. train_loader = DataLoader du train (lots de BATCH_SIZE, mélangés)
# 2. val_loader = DataLoader du val (lots de BATCH_SIZE, pas mélangés)
# 3. affiche la forme du premier lot du train_loader (entrées et étiquettes)

In [ ]:
# ✅ Contrôle
xb, yb = next(iter(train_loader))
assert xb.shape == (BATCH_SIZE, MAX_LEN) and yb.shape == (BATCH_SIZE,)
assert len(train_loader) == 57 if BATCH_SIZE == 64 else True, "3617 SMS / 64 = 57 lots (le dernier est incomplet)"
assert len(val_loader.dataset) == 776
print("✅ étape 4")

---
# Étape 5 — Le modèle `SpamNet`

🎯 Écrire la classe `SpamNet` : **c'est `SentimentNet` de la leçon (cellule 11)**, renommée. Relis-la ligne à ligne :

```
SMS (64 SMS × MAX_LEN numéros)
  └─ nn.Embedding   → un vecteur de EMBEDDING_DIM nombres par mot        (64, MAX_LEN, 32)
  └─ somme ÷ nombre de vrais mots (mean pooling, les 0 ne comptent pas) (64, 32)
  └─ nn.Linear(32, 1) → un seul nombre par SMS : le logit                 (64,)
       logit > 0 → spam, logit < 0 → ham
```

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `nn.Embedding(vocab_size, dim, padding_idx=0)` | la table des vecteurs de mots | `padding_idx=0` : la ligne `<pad>` reste à zéro | 📍 M05-D05 cellule 11 |
| `.sum(dim=1)`, `(x != 0).sum(dim=1, keepdim=True).clamp(min=1)` | somme des vecteurs, nombre de vrais mots | — | 📍 M05-D05 cellule 11 |
| `nn.Linear(dim, 1)` puis `.squeeze(1)` | un logit par SMS, forme `(batch,)` | — | 📍 M05-D05 cellule 11 |
| `.to(device)` | envoyer le modèle sur le GPU | — | 📍 M05-D03 |

Noms imposés (pour l'étape 11) : `self.embedding` et `self.classifier`.

In [ ]:
# ✍️ À toi
# 1. class SpamNet(nn.Module) : __init__(self, vocab_size, embedding_dim) et forward(self, x), comme SentimentNet
# 2. model = SpamNet(vocab_size, EMBEDDING_DIM), envoyé sur device ; affiche-le

In [ ]:
# ✅ Contrôle
with torch.no_grad():
    sortie = model(X_val[:8].to(device))
assert sortie.shape == (8,), f"un logit par SMS attendu : forme (8,), obtenue {tuple(sortie.shape)}"
n_params = sum(p.numel() for p in model.parameters())
assert n_params == vocab_size * EMBEDDING_DIM + EMBEDDING_DIM + 1, n_params
assert hasattr(model, "embedding") and hasattr(model, "classifier")
print(f"✅ étape 5 : {n_params:,} paramètres".replace(",", " "))

✍️ **À écrire** : presque tous ces paramètres sont dans une seule couche. Laquelle, et pourquoi ?

*Ta réponse :*

---
# Étape 6 — Entraîner, en surveillant le val

🎯 Écrire `train_model`, en partant de **la fonction du même nom de M05-D03** (cellule « def train_model »), avec 3 adaptations :

| Dans M05-D03 (iris, 3 classes) | Ici (spam, 2 classes, **un seul logit**) |
|---|---|
| `outputs.argmax(dim=1)` pour la classe prédite | `outputs > 0` (logit positif = spam) |
| `CrossEntropyLoss` | `BCEWithLogitsLoss` (comme la leçon M05-D05) |
| garde le modèle de la **dernière** époque | garde le modèle de **la meilleure époque** (la plus petite `val_loss`), comme `best_state` dans M05-D04 Transfer learning |

Pourquoi garder la meilleure époque ? Avec 344 000 paramètres pour 3 617 SMS, le modèle finit par **apprendre le train par cœur** : la loss du train continue de baisser, celle du val remonte (l'overfitting de M04-D03).

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `model.train()` / `model.eval()` + `torch.no_grad()` | mode entraînement / mode évaluation sans gradient | — | 📍 M05-D03 `train_model` |
| `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` | les 3 temps d'une mise à jour | — | 📍 M05-D03, M05-D05 |
| `nn.BCEWithLogitsLoss()` | loss binaire sur un logit brut (sigmoïde incluse) | — | 📍 M05-D05 cellule 14 |
| `optim.Adam(model.parameters(), lr=...)` | l'optimiseur | `lr=LEARNING_RATE` | 📍 M05-D05 |
| `copy.deepcopy(model.state_dict())` | une **copie** des poids, qui ne bougera plus | — | 📍 M05-D04 Transfer learning (`best_state`) |
| `model.load_state_dict(best_state)` | remet les poids copiés | — | 📍 M05-D04, M05-D03 |

In [ ]:
# ✍️ À toi
# 1. criterion = BCEWithLogitsLoss ; optimizer = Adam sur les paramètres de model
# 2. def train_model(model, train_loader, val_loader, criterion, optimizer, epochs):
#      history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
#      best_val_loss = un très grand nombre ; best_state = None
#      pour chaque époque :
#         phase train : comme M05-D03 (inputs/labels sur device, les 3 temps, cumul loss et bonnes réponses)
#         phase val   : comme M05-D03, sans gradient
#         ranger les 4 moyennes dans history, afficher une ligne par époque
#         si la val_loss est la meilleure vue : la retenir + copier les poids dans best_state
#      remettre best_state dans le modèle, afficher la meilleure époque
#      return history
# 3. history = train_model(..., epochs=EPOCHS)

In [ ]:
# ✅ Contrôle
assert len(history["val_loss"]) == EPOCHS and len(history["train_loss"]) == EPOCHS
assert history["train_loss"][-1] < history["train_loss"][0], "la loss du train doit baisser"
assert min(history["val_loss"]) < 0.25, f"meilleure val_loss : {min(history['val_loss']):.3f}"
print(f"✅ étape 6 : meilleure époque {history['val_loss'].index(min(history['val_loss'])) + 1}, val_loss {min(history['val_loss']):.4f}")

---
# Étape 7 — Lire les courbes

🎯 Tracer la loss du train et celle du val, époque par époque, comme le « Plot 1 » de M05-D03.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `plt.plot(liste, label=...)` puis `plt.legend()` | une courbe par liste | `label="Training Loss"` | 📍 M05-D03 Neural networks with PyTorch (cellule « Plot 1 ») |

In [ ]:
# ✍️ À toi
# 1. courbes train_loss et val_loss sur le même graphique, avec titre, axes et légende
# 2. (facultatif) pareil pour train_acc et val_acc

✍️ **À écrire** : à partir de quelle époque la loss du val arrête de baisser ? Que fait celle du train pendant ce temps, et comment s'appelle ce phénomène ?

*Ta réponse :*

---
# Étape 8 — Évaluer sur le val

🎯 Calculer, sur le **val**, la probabilité de spam de chaque SMS, puis la **précision**, le **rappel** et le **F1**, et afficher la matrice de confusion.

Rappel du matin : **précision** = parmi les SMS classés spam, combien en sont vraiment ; c'est elle qui compte les vrais messages bloqués à tort.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `torch.sigmoid(logits)` | logit → probabilité entre 0 et 1 | — | 📍 M05-D05 cellule 20 (`predict`) |
| `.cpu()` | ramène un tenseur du GPU vers le processeur (sklearn ne lit pas le GPU) | — | 🆕 le retour de `.to(device)` |
| `(probas > 0.5).int()` | probabilité → prédiction 0 / 1 | le seuil 0,5 | 📍 M05-D05 cellule 17 (`> 0`) |
| `f1_score(vrai, prédit)` | le F1 | — | 📍 M04-D04 Decision trees template |
| `precision_score`, `recall_score` | précision, rappel | mêmes arguments que `f1_score` | 🆕 les jumeaux de `f1_score`, même import |
| `ConfusionMatrixDisplay.from_predictions(vrai, prédit)` | la matrice de confusion en image | — | 📍 M04-D04 (`ConfusionMatrixDisplay`) + 🆕 `from_predictions` |

In [ ]:
# ✍️ À toi
# 1. en mode évaluation et sans gradient : val_probas = sigmoïde des logits de model sur X_val (envoyé sur device), ramenée sur cpu
# 2. val_preds = 1 si val_probas > 0.5, sinon 0
# 3. affiche précision, rappel et F1 sur le val
# 4. affiche la matrice de confusion

In [ ]:
# ✅ Contrôle
assert val_probas.shape == (776,) and 0 <= val_probas.min() and val_probas.max() <= 1
f1_val = f1_score(y_val, val_preds)
assert f1_val > 0.75, f"F1 val : {f1_val:.3f}"
print(f"✅ étape 8 : F1 val {f1_val:.3f} (UNK_ID = {UNK_ID})")

✍️ **Note tes résultats** (tu rempliras la 2ᵉ ligne à l'étape 9) :

| Réglage | Précision val | Rappel val | F1 val | Vrais messages bloqués (faux positifs) | Spams ratés (faux négatifs) |
|---|---|---|---|---|---|
| `UNK_ID = 1` (comme la leçon) | | | | | |
| `UNK_ID = 0` | | | | | |

---
# Étape 9 — L'expérience `<unk>` : un piège que la leçon ne montre pas

**Le problème.** `nn.Embedding` donne à chaque ligne de sa table des nombres **au hasard** au départ ; l'entraînement les corrige ensuite. Mais une ligne n'est corrigée que si son mot **apparaît dans le train**.

Or la ligne `1` (`<unk>`) n'apparaît **jamais** dans le train : le vocabulaire vient du train, donc aucun mot du train n'est inconnu (le contrôle de l'étape 3 le vérifie). Son vecteur reste au hasard. Et dans le val, **1 mot sur 8** tombe sur ce vecteur au hasard, qui est mélangé à la moyenne des vrais mots.

Dans la leçon, ça ne se voyait pas : les 6 avis du test n'avaient aucun mot inconnu.

**L'expérience.** Une seule ligne à changer :
1. dans la cellule des réglages (étape 0), mets `UNK_ID = 0` : un mot inconnu est alors traité comme du padding, et **ignoré** dans la moyenne ;
2. menu **Run All** (tout relancer depuis le début) ;
3. remplis la 2ᵉ ligne du tableau de l'étape 8.

🎯 Ce que tu compares : la **précision** surtout. Prédis avant de lancer : monte-t-elle, baisse-t-elle ?

✍️ **À écrire (2-3 phrases)** : quel réglage gardes-tu, pourquoi, et qu'est-ce qu'on perd en ignorant les mots inconnus ?

*Ta réponse :*

<details><summary>🆕 Pour aller plus loin (hors cours) : une autre façon de corriger</summary>

Au lieu d'ignorer les mots inconnus, on peut **entraîner** la ligne `<unk>` : dans le vocabulaire, on ne garde que les mots vus **au moins 2 fois** dans le train. Les mots vus une seule fois deviennent `<unk>` dès le train, et sa ligne apprend « un mot rare ». Il faut compter les mots avec un dictionnaire (`comptes[w] = comptes.get(w, 0) + 1`), puis filtrer.
</details>

---
# Étape 10 — Lire les erreurs

🎯 Lire les **faux positifs** (vrais messages classés spam : l'erreur la plus grave pour AT&T) et les **faux négatifs** (spams ratés). Les chiffres disent combien ; les messages disent **pourquoi**.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `val_df["proba"] = val_probas.numpy()` | range les probabilités à côté des textes (même ordre : `shuffle=False`) | — | 📍 M05-D03 (`.numpy()`) |
| `df[(cond1) & (cond2)]` | les lignes qui remplissent **les deux** conditions | chaque condition entre parenthèses | 🆕 deux conditions à la fois |
| `.sort_values("proba", ascending=False)` | trie, les plus sûrs en premier | — | 📍 M02-D02 Data Transformation |

In [ ]:
# ✍️ À toi
# 1. ajoute la colonne proba à val_df
# 2. faux_positifs = les ham (label 0) avec proba > 0.5, triés du plus sûr au moins sûr ; affiche text et proba
# 3. faux_negatifs = les spams (label 1) avec proba <= 0.5 ; affiche text et proba

✍️ **À écrire** : qu'ont en commun les faux positifs ? Et les spams ratés ? (longueur, chiffres, mots…)

*Ta réponse :*

---
# Étape 11 — 🆕 Bonus : quels mots poussent vers spam ou ham ?

Le modèle fait la moyenne des vecteurs des mots, puis applique `classifier`. Si on applique `classifier` au vecteur **d'un seul mot**, on obtient « le score de ce mot » : très positif = mot de spam, très négatif = mot de ham.

`model.classifier(model.embedding.weight)` calcule ce score pour **tous les mots d'un coup** (une ligne par mot). Il reste à trier et à retrouver les mots grâce au dictionnaire inverse `id_to_word` (comme `itos` dans M06-D01).

C'est la vérification de notre discussion sur `&lt;#&gt;` : arrive-t-il dans les mots les plus « ham » ?

In [ ]:
# 🆕 Bonus
# 1. id_to_word = le dictionnaire inverse de word_to_id
# 2. sans gradient : scores = classifier appliqué à tous les vecteurs de mots, en une liste (une valeur par mot)
# 3. affiche les 15 mots au score le plus haut (spam) et les 15 au score le plus bas (ham)
# 4. à quel rang arrive "&lt;#&gt;" côté ham ?

---
# Étape 12 — Enregistrer le modèle et ses probabilités

🎯 Deux sauvegardes :
1. **le modèle**, avec tout ce qu'il faut pour le réutiliser : ses poids, mais aussi le vocabulaire et les réglages (sans `word_to_id`, impossible d'encoder un nouveau SMS) ;
2. **ses probabilités sur le val et le test**, dans un CSV que le template 4 utilisera pour comparer les modèles. Tu calcules celles du test, mais tu ne regardes **pas** ses scores ici.

🧰 Outils

| Outil | Rôle | Arguments importants | Vu où |
|---|---|---|---|
| `torch.save(dictionnaire, chemin)` | enregistre un dictionnaire de poids et d'infos | les clés : `"model_state_dict"`, `"word_to_id"`, `"max_len"`, `"unk_id"`, `"embedding_dim"` | 📍 M05-D03 (`checkpoint_data`) |
| `pd.DataFrame({...})` | fabriquer un tableau à partir d'un dictionnaire | — | 📍 M02-D01 Data Generation |
| `pd.concat([df1, df2])` | empiler deux tableaux | `ignore_index=True` | 📍 M05-D01 Gradient descent, M05-D04 CNN with PyTorch |
| `.to_csv(chemin, index=False)` | écrit le CSV | — | 📍 M02-D01 |

In [ ]:
# ✍️ À toi
# 1. torch.save du dictionnaire (poids + word_to_id + max_len + unk_id + embedding_dim) dans DOSSIER_MODELES / "modele2_embeddings.pt"
# 2. test_probas = probabilités du modèle sur X_test (comme l'étape 8)
# 3. probas_df = un tableau avec les colonnes split ("val" ou "test"), text, label, proba : les 776 SMS du val puis les 776 du test
# 4. écris-le dans DOSSIER_SRC / "probas_modele2.csv"

In [ ]:
# ✅ Contrôle
relu = pd.read_csv(DOSSIER_SRC / "probas_modele2.csv")
assert list(relu.columns) == ["split", "text", "label", "proba"], list(relu.columns)
assert (relu["split"] == "val").sum() == 776 and (relu["split"] == "test").sum() == 776
assert relu["proba"].between(0, 1).all()
sauvegarde = torch.load(DOSSIER_MODELES / "modele2_embeddings.pt", weights_only=True)
assert {"model_state_dict", "word_to_id", "max_len", "unk_id", "embedding_dim"} <= set(sauvegarde)
print("✅ étape 12 : modèle et probabilités enregistrés pour le template 4")

---
# 📝 Pour le livrable

| Élément | Ce que tu écris |
|---|---|
| Architecture | Embedding (dim ?) → mean pooling → Linear(?, 1) ; combien de paramètres ? |
| Tokenisation | minuscules + découpage aux espaces, vocabulaire du train seulement (taille ?) |
| `MAX_LEN` | |
| Mots inconnus | `UNK_ID` choisi et pourquoi (tableau de l'étape 8) |
| Entraînement | Adam, lr ?, lots de ?, meilleure époque sur la val_loss |
| Scores **val** | précision / rappel / F1 |
| Limites | ce que les erreurs de l'étape 10 montrent ; `&lt;#&gt;` (étape 11) |

Les scores du **test** viendront dans le template 4.